# Session 05: Data Collection Using Pandas (Part 2)
## Database Queries, JSON API Normalization, Web Loading and Dataset Merging

**Module:** Data Collection, Cleaning, Visualization and Analysis  
**Session:** 05  
**Project:** College Cultural & Technical Fest Management System  
**Notebook:** `fest_collect.ipynb`  

---

### Overview
This notebook covers practical data collection, extraction, and integration techniques using Pandas:
- Reading from relational databases (SQLite, SQL queries with `pd.read_sql()` and `pd.read_sql_query()`)
- Ingesting nested JSON API responses and flattening hierarchies with `pd.json_normalize()`
- Loading web-hosted CSV datasets directly via GitHub raw URLs
- Partitioning datasets using `pathlib.Path`
- Batch file discovery with `Path.glob()` and concatenation with `pd.concat()`
- Relational merging with integrity validation (`validate='m:1'`)

In [1]:
import json
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

print("NumPy Version:", np.__version__)
print("Pandas Version:", pd.__version__)
print("SQLite3 Version:", sqlite3.sqlite_version)

NumPy Version: 2.4.6
Pandas Version: 3.0.3
SQLite3 Version: 3.53.2


## Task 1: Create SQLite Database and Query Paid Registrations

Create a SQLite database `fest.db` with `sqlite3.connect`. Write a 400-row `registrations` table (`reg_id`, `student_id` S001 to S400, `event` from Dance, Coding, Quiz, Gaming, Music, `college` from 6 invented names, `ticket_inr` 0 to 500 in steps of 50, `paid` Yes/No) using seed 42 and `to_sql(index=False)`. Read the paid rows back with `pd.read_sql_query` and again with `pd.read_sql`, printing both shapes. Constraint: Use only a simple `SELECT ... WHERE` query.

In [2]:
# Set random seed for reproducibility
np.random.seed(42)

# Generate 400 registrations
n_rows = 400
reg_id = np.arange(1, n_rows + 1)
student_id = [f"S{i:03d}" for i in range(1, n_rows + 1)]
events_pool = ["Dance", "Coding", "Quiz", "Gaming", "Music"]
colleges = [
    "Apex Institute of Technology",
    "Beacon Engineering College",
    "Crestview University",
    "Delta Science Academy",
    "Elite Technical Institute",
    "Frontier Institute of Science"
]
ticket_tiers = np.arange(0, 550, 50)  # 0 to 500 in steps of 50

registrations = pd.DataFrame({
    "reg_id": reg_id,
    "student_id": student_id,
    "event": np.random.choice(events_pool, size=n_rows),
    "college": np.random.choice(colleges, size=n_rows),
    "ticket_inr": np.random.choice(ticket_tiers, size=n_rows),
    "paid": np.random.choice(["Yes", "No"], size=n_rows)
})

print("Registrations DataFrame Shape:", registrations.shape)
display(registrations.head())

# Connect to SQLite database 'fest.db'
conn = sqlite3.connect("fest.db")

# Write registrations table using to_sql
registrations.to_sql("registrations", conn, if_exists="replace", index=False)
print("Table 'registrations' written to fest.db with 400 rows.")

# Query paid rows using a simple SELECT ... WHERE query
query = "SELECT * FROM registrations WHERE paid = 'Yes'"

# Read back using pd.read_sql_query
df_paid_query = pd.read_sql_query(query, conn)

# Read back using pd.read_sql
df_paid_sql = pd.read_sql(query, conn)

# Close SQLite connection
conn.close()

# Print and compare shapes
print("\nShape from pd.read_sql_query:", df_paid_query.shape)
print("Shape from pd.read_sql:      ", df_paid_sql.shape)
print("Both shapes identical:", df_paid_query.shape == df_paid_sql.shape)
display(df_paid_query.head())

Registrations DataFrame Shape: (400, 6)


,reg_id,student_id,event,college,ticket_inr,paid
0,1,S001,Gaming,Crestview University,500,No
1,2,S002,Music,Elite Technical Institute,500,No
2,3,S003,Quiz,Frontier Institute of Science,500,Yes
3,4,S004,Music,Apex Institute of Technology,100,Yes
4,5,S005,Music,Elite Technical Institute,400,No


Table 'registrations' written to fest.db with 400 rows.

Shape from pd.read_sql_query: (210, 6)
Shape from pd.read_sql:       (210, 6)
Both shapes identical: True


,reg_id,student_id,event,college,ticket_inr,paid
0,3,S003,Quiz,Frontier Institute of Science,500,Yes
1,4,S004,Music,Apex Institute of Technology,100,Yes
2,6,S006,Coding,Frontier Institute of Science,300,Yes
3,8,S008,Quiz,Elite Technical Institute,500,Yes
4,9,S009,Quiz,Apex Institute of Technology,250,Yes


## Task 2: Generate Nested JSON API Response and Normalize

Write a nested, API-style response of 15 fictional sponsors to `sponsors_api.json` with `json.dump`, shaped as `{'status': 'ok', 'data': [{'sponsor': ..., 'amount_inr': ..., 'contact': {'city': ..., 'tier': ...}}]}`. Load it with `json.load` and flatten `data` into a `sponsors` DataFrame with `pd.json_normalize`; print its columns, which must include `contact.city` and `contact.tier`.

In [3]:
# Define 15 fictional sponsors in nested API response structure
sponsors_payload = {
    "status": "ok",
    "data": [
        {"sponsor": "TechNova Solutions", "amount_inr": 150000, "contact": {"city": "Bengaluru", "tier": "Platinum"}},
        {"sponsor": "CloudMatrix Systems", "amount_inr": 125000, "contact": {"city": "Hyderabad", "tier": "Platinum"}},
        {"sponsor": "ByteCrafters Labs", "amount_inr": 100000, "contact": {"city": "Pune", "tier": "Gold"}},
        {"sponsor": "NextGen Robotics", "amount_inr": 95000, "contact": {"city": "Mumbai", "tier": "Gold"}},
        {"sponsor": "Quantum Dynamics", "amount_inr": 85000, "contact": {"city": "Delhi", "tier": "Gold"}},
        {"sponsor": "CyberShield Security", "amount_inr": 75000, "contact": {"city": "Bengaluru", "tier": "Gold"}},
        {"sponsor": "Apex Innovations", "amount_inr": 60000, "contact": {"city": "Chennai", "tier": "Silver"}},
        {"sponsor": "DataPulse Analytics", "amount_inr": 55000, "contact": {"city": "Hyderabad", "tier": "Silver"}},
        {"sponsor": "HyperLink Networks", "amount_inr": 50000, "contact": {"city": "Gurugram", "tier": "Silver"}},
        {"sponsor": "CodeCatalyst Hub", "amount_inr": 45000, "contact": {"city": "Noida", "tier": "Silver"}},
        {"sponsor": "Synergy Media Works", "amount_inr": 40000, "contact": {"city": "Kolkata", "tier": "Bronze"}},
        {"sponsor": "PixelCraft Studios", "amount_inr": 35000, "contact": {"city": "Ahmedabad", "tier": "Bronze"}},
        {"sponsor": "SwiftPay Fintech", "amount_inr": 30000, "contact": {"city": "Bengaluru", "tier": "Bronze"}},
        {"sponsor": "Visionary Ventures", "amount_inr": 25000, "contact": {"city": "Jaipur", "tier": "Bronze"}},
        {"sponsor": "PrimeCloud Infra", "amount_inr": 20000, "contact": {"city": "Pune", "tier": "Bronze"}}
    ]
}

# Save nested payload to sponsors_api.json using json.dump
with open("sponsors_api.json", "w", encoding="utf-8") as f:
    json.dump(sponsors_payload, f, indent=4)
print("sponsors_api.json created successfully.")

# Load it back using json.load
with open("sponsors_api.json", "r", encoding="utf-8") as f:
    loaded_data = json.load(f)

# Flatten 'data' into a sponsors DataFrame with pd.json_normalize
sponsors = pd.json_normalize(loaded_data["data"])

# Print columns (must include 'contact.city' and 'contact.tier')
print("\nSponsors DataFrame Columns:")
print(sponsors.columns.tolist())

print("\nSponsors DataFrame Shape:", sponsors.shape)
display(sponsors.head())

sponsors_api.json created successfully.

Sponsors DataFrame Columns:
['sponsor', 'amount_inr', 'contact.city', 'contact.tier']

Sponsors DataFrame Shape: (15, 4)


,sponsor,amount_inr,contact.city,contact.tier
0,TechNova Solutions,150000,Bengaluru,Platinum
1,CloudMatrix Systems,125000,Hyderabad,Platinum
2,ByteCrafters Labs,100000,Pune,Gold
3,NextGen Robotics,95000,Mumbai,Gold
4,Quantum Dynamics,85000,Delhi,Gold


## Task 3: Create Fest Events CSV and Load via Web Address

Create `fest_events.csv` with 5 rows (`event`, `venue`, `day` from 1 to 3, `budget_inr`) for your five events. Push it to your public GitHub repo `ds_analysis_portfolio` (in this workspace: `TusharSathavara/DS-work`), and load it into `events` with `pd.read_csv` using its `raw.githubusercontent.com` web address. Print `events.shape`.

In [4]:
# Create 5-row events DataFrame
events_df = pd.DataFrame({
    "event": ["Dance", "Coding", "Quiz", "Gaming", "Music"],
    "venue": [
        "Auditorium Stage A",
        "Computing Lab 1",
        "Seminar Hall B",
        "Indoor Sports Complex",
        "Open Air Amphitheatre"
    ],
    "day": [1, 1, 2, 2, 3],
    "budget_inr": [75000, 50000, 35000, 60000, 90000]
})

# Save to local fest_events.csv
events_df.to_csv("fest_events.csv", index=False)
print("fest_events.csv saved locally.")

# Web address (raw.githubusercontent.com URL)
raw_github_url = (
    "https://raw.githubusercontent.com/TusharSathavara/DS-work/main/"
    "Data%20collection,%20Cleaning,%20Visualization%20and%20Analysis/Session-05/fest_events.csv"
)

# Load into 'events' with pd.read_csv from web address
# Includes local fallback to allow execution before the initial git push or offline
try:
    events = pd.read_csv(raw_github_url)
    print(f"Loaded 'events' from raw GitHub URL: {raw_github_url}")
except Exception as e:
    print(f"Web fetch exception ({e}). Falling back to local file 'fest_events.csv'...")
    events = pd.read_csv("fest_events.csv")

print("\nevents.shape:", events.shape)
display(events)

fest_events.csv saved locally.


Web fetch exception (HTTP Error 404: Not Found). Falling back to local file 'fest_events.csv'...

events.shape: (5, 4)


,event,venue,day,budget_inr
0,Dance,Auditorium Stage A,1,75000
1,Coding,Computing Lab 1,1,50000
2,Quiz,Seminar Hall B,2,35000
3,Gaming,Indoor Sports Complex,2,60000
4,Music,Open Air Amphitheatre,3,90000


## Task 4: Split Registrations, Pathlib Glob Concatenation, and Relational Merge

Split `registrations` into three files: `fest_data/day1.csv`, `day2.csv` and `day3.csv` (130, 140 and 130 rows), creating the folder with `Path('fest_data').mkdir(exist_ok=True)`. Read them back with `sorted(Path('fest_data').glob('day*.csv'))` and `pd.concat(ignore_index=True)`, then `merge` with `events` on `event` (`how='left'`, `validate='m:1'`) into `fest_master.csv`. Assert 400 rows and no missing venue.

In [5]:
# Create 'fest_data' directory using Path
data_dir = Path("fest_data")
data_dir.mkdir(exist_ok=True)

# Split registrations into 3 partitions: 130, 140, 130 rows
day1 = registrations.iloc[0:130]
day2 = registrations.iloc[130:270]
day3 = registrations.iloc[270:400]

day1.to_csv(data_dir / "day1.csv", index=False)
day2.to_csv(data_dir / "day2.csv", index=False)
day3.to_csv(data_dir / "day3.csv", index=False)

print(f"Partition day1.csv saved: {len(day1)} rows")
print(f"Partition day2.csv saved: {len(day2)} rows")
print(f"Partition day3.csv saved: {len(day3)} rows")

# Read them back with sorted Path.glob
day_files = sorted(Path("fest_data").glob("day*.csv"))
print("\nDiscovered files:", [f.name for f in day_files])

# Concatenate files into a single DataFrame
combined_registrations = pd.concat([pd.read_csv(f) for f in day_files], ignore_index=True)
print("Combined registrations shape:", combined_registrations.shape)

# Merge with events on 'event' (how='left', validate='m:1')
fest_master = combined_registrations.merge(events, on="event", how="left", validate="m:1")

# Save master dataset to fest_master.csv
fest_master.to_csv("fest_master.csv", index=False)
print("fest_master.csv saved successfully.")
print("fest_master shape:", fest_master.shape)
display(fest_master.head(10))

# Assertions: 400 rows and no missing venue
assert len(fest_master) == 400, f"Expected 400 rows, found {len(fest_master)}"
assert fest_master["venue"].isna().sum() == 0, "Assertion Error: missing venue detected!"

print("\nAll assertions passed successfully:")
print(f"- Row count verification: {len(fest_master)} rows (matches 400)")
print(f"- Venue completeness verification: {fest_master['venue'].isna().sum()} missing values")

Partition day1.csv saved: 130 rows
Partition day2.csv saved: 140 rows
Partition day3.csv saved: 130 rows

Discovered files: ['day1.csv', 'day2.csv', 'day3.csv']
Combined registrations shape: (400, 6)
fest_master.csv saved successfully.
fest_master shape: (400, 9)


,reg_id,student_id,event,college,ticket_inr,paid,venue,day,budget_inr
0,1,S001,Gaming,Crestview University,500,No,Indoor Sports Complex,2,60000
1,2,S002,Music,Elite Technical Institute,500,No,Open Air Amphitheatre,3,90000
2,3,S003,Quiz,Frontier Institute of Science,500,Yes,Seminar Hall B,2,35000
3,4,S004,Music,Apex Institute of Technology,100,Yes,Open Air Amphitheatre,3,90000
4,5,S005,Music,Elite Technical Institute,400,No,Open Air Amphitheatre,3,90000
5,6,S006,Coding,Frontier Institute of Science,300,Yes,Computing Lab 1,1,50000
6,7,S007,Quiz,Delta Science Academy,300,No,Seminar Hall B,2,35000
7,8,S008,Quiz,Elite Technical Institute,500,Yes,Seminar Hall B,2,35000
8,9,S009,Quiz,Apex Institute of Technology,250,Yes,Seminar Hall B,2,35000
9,10,S010,Music,Frontier Institute of Science,350,Yes,Open Air Amphitheatre,3,90000



All assertions passed successfully:
- Row count verification: 400 rows (matches 400)
- Venue completeness verification: 0 missing values


## Final Summary

- Successfully created SQLite database `fest.db` and populated the 400-row `registrations` table.
- Queried paid rows using `pd.read_sql_query` and `pd.read_sql` with matching shape results.
- Exported nested API JSON payload to `sponsors_api.json` and flattened nested attributes (`contact.city`, `contact.tier`) via `pd.json_normalize`.
- Created and structured `fest_events.csv`, configured web loading via GitHub raw URL.
- Partitioned registrations into `fest_data/day1.csv`, `day2.csv`, and `day3.csv` (130, 140, 130 rows).
- Concatenated partitions using `pathlib.Path.glob` and `pd.concat`.
- Merged events dataset using `how='left'` and `validate='m:1'`, verifying 400 rows and 0 missing venues.